# Phase 3 Step 2: Audio Feature Engineering & Preprocessing

## Overview
This notebook demonstrates comprehensive audio feature extraction for deepfake detection.

**Context:** Phase 2 analysis revealed that 100% of high-risk misinformation cases involve audio or video beyond text detection.

**Objective:** Extract 35+ audio features including MFCCs, spectral features, and prosodic characteristics to enable ML-based voice forensics.

**Deliverables:**
- Feature extraction pipeline (MFCC, spectral, prosodic, temporal)
- Audio preprocessing utilities (normalization, silence removal, augmentation)
- EDA showcasing feature distributions (genuine vs. deepfake)
- Feature matrix ready for baseline model training (Step 3)

## Section 1: Import Required Libraries and Dependencies

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Audio processing
import librosa
import librosa.display
from scipy import signal, stats

# Plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

# Add src to path for custom modules
sys.path.insert(0, '../../src')
from audio_features import AudioFeatureExtractor, AudioPreprocessor

print("✓ All libraries imported successfully")

## Section 2: Load and Preprocess Audio Data

In [ ]:
# Configuration
SAMPLE_RATE = 16000  # Standard for speech processing
N_MFCC = 13          # Mel-frequency cepstral coefficients

# Paths (using mock data for demonstration)
AUDIO_DATA_DIR = Path('../../data/audio_samples')
OUTPUT_DIR = Path('../../outputs/audio_features')

AUDIO_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Audio directory: {AUDIO_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Create demonstration audio samples (genuine vs synthetic)
def generate_synthetic_speech(duration=3, sr=16000, label='genuine'):
    """Generate synthetic audio samples for demonstration."""
    t = np.linspace(0, duration, int(sr * duration))
    
    if label == 'genuine':
        # Natural speech-like: multiple frequency components
        y = (0.3 * np.sin(2 * np.pi * 150 * t) +  # Fundamental
             0.15 * np.sin(2 * np.pi * 300 * t) +  # First harmonic
             0.1 * np.sin(2 * np.pi * 450 * t))    # Second harmonic
        # Add natural variation
        y += 0.02 * np.random.randn(len(y))
        # Amplitude envelope (prosodic variation)
        envelope = np.sin(2 * np.pi * 1.5 * t) ** 2
        y *= envelope
    else:
        # Synthetic/deepfake: more artificial (flat spectrum)
        y = 0.3 * signal.square(2 * np.pi * 150 * t)  # Square wave
        y += 0.05 * np.random.randn(len(y))
        # No prosodic variation
        y *= 0.5  # Flatten amplitude
    
    return y

# Create sample audio files
for idx in range(5):
    # Genuine samples
    y_genuine = generate_synthetic_speech(duration=3, label='genuine')
    path_gen = AUDIO_DATA_DIR / f'genuine_{idx:02d}.wav'
    librosa.output.write_wav(str(path_gen), y_genuine, SAMPLE_RATE)
    
    # Deepfake samples
    y_fake = generate_synthetic_speech(duration=3, label='deepfake')
    path_fake = AUDIO_DATA_DIR / f'deepfake_{idx:02d}.wav'
    librosa.output.write_wav(str(path_fake), y_fake, SAMPLE_RATE)

audio_files = sorted(list(AUDIO_DATA_DIR.glob('*.wav')))
print(f"✓ Created {len(audio_files)} sample audio files")
print(f"  Sample files: {[f.name for f in audio_files[:3]]}")

In [ ]:
# Initialize preprocessing pipeline
preprocessor = AudioPreprocessor(sr=SAMPLE_RATE)

# Check audio quality
print("Audio Quality Checks:")
print("-" * 50)
quality_results = []

for audio_file in audio_files[:3]:  # Check first 3 files
    quality = preprocessor.check_audio_quality(str(audio_file))
    quality_results.append(quality)
    status = "✓" if quality['valid'] else "✗"
    print(f"{status} {audio_file.name:25s} | {quality['reason']}")

print(f"\n✓ Quality check completed for {len(quality_results)} files")

## Section 3: Extract Audio Features (MFCCs, Spectral, Prosodic)

In [ ]:
# Initialize feature extractor
extractor = AudioFeatureExtractor(sr=SAMPLE_RATE, n_mfcc=N_MFCC)

# Extract features from all audio files
print("Extracting audio features...")
features_list = []

for idx, audio_file in enumerate(audio_files):
    features = extractor.extract_all_features(str(audio_file))
    if features:
        features['file'] = audio_file.name
        features['label'] = 'genuine' if 'genuine' in audio_file.name else 'deepfake'
        features_list.append(features)
    
    if (idx + 1) % 5 == 0:
        print(f"  Processed {idx + 1}/{len(audio_files)} files")

# Create feature DataFrame
df_features = pd.DataFrame(features_list)
print(f"\n✓ Extracted {len(df_features)} audio feature sets")
print(f"  Total features per file: {len(df_features.columns) - 2}")
print(f"\nFeature DataFrame shape: {df_features.shape}")
print(f"\nFirst 5 rows (sample):")
print(df_features.iloc[:5, :10])  # Show first 10 columns

## Section 4: Exploratory Data Analysis - Feature Distributions

In [ ]:
# EDA: MFCC Features Comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('MFCC Features: Genuine vs Deepfake Audio', fontsize=14, fontweight='bold')

mfcc_features = [f for f in df_features.columns if f.startswith('mfcc_') and f.endswith('_mean')]

for idx, feature in enumerate(mfcc_features[:6]):
    ax = axes[idx // 3, idx % 3]
    
    genuine_data = df_features[df_features['label'] == 'genuine'][feature]
    fake_data = df_features[df_features['label'] == 'deepfake'][feature]
    
    ax.hist(genuine_data, alpha=0.6, label='Genuine', bins=8, color='blue')
    ax.hist(fake_data, alpha=0.6, label='Deepfake', bins=8, color='red')
    ax.set_xlabel(feature.replace('_mean', ''), fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mfcc_distribution_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ MFCC distribution plot saved")

In [ ]:
# EDA: Spectral Features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Spectral Features: Genuine vs Deepfake', fontsize=14, fontweight='bold')

spectral_features = ['spectral_centroid_mean', 'spectral_rolloff_mean', 'spectral_bandwidth_mean']
feature_labels = ['Spectral Centroid', 'Spectral Rolloff', 'Spectral Bandwidth']

for idx, (feature, label) in enumerate(zip(spectral_features, feature_labels)):
    ax = axes[idx]
    
    genuine_data = df_features[df_features['label'] == 'genuine'][feature]
    fake_data = df_features[df_features['label'] == 'deepfake'][feature]
    
    data_to_plot = [genuine_data, fake_data]
    bp = ax.boxplot(data_to_plot, labels=['Genuine', 'Deepfake'], patch_artist=True)
    
    colors = ['lightblue', 'lightcoral']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    ax.set_ylabel(label, fontsize=11)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'spectral_features_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Spectral features boxplot saved")

In [ ]:
# EDA: Prosodic Features (F0 and Voicing)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Prosodic Features: Genuine vs Deepfake', fontsize=14, fontweight='bold')

# F0 Mean Distribution
ax = axes[0]
genuine_f0 = df_features[df_features['label'] == 'genuine']['f0_mean']
fake_f0 = df_features[df_features['label'] == 'deepfake']['f0_mean']

ax.hist(genuine_f0, alpha=0.6, label='Genuine', bins=8, color='blue')
ax.hist(fake_f0, alpha=0.6, label='Deepfake', bins=8, color='red')
ax.set_xlabel('Fundamental Frequency (Hz)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)

# Voicing Ratio Distribution
ax = axes[1]
genuine_voicing = df_features[df_features['label'] == 'genuine']['voicing_ratio']
fake_voicing = df_features[df_features['label'] == 'deepfake']['voicing_ratio']

ax.hist(genuine_voicing, alpha=0.6, label='Genuine', bins=8, color='blue')
ax.hist(fake_voicing, alpha=0.6, label='Deepfake', bins=8, color='red')
ax.set_xlabel('Voicing Ratio', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prosodic_features_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Prosodic features distribution plot saved")

In [ ]:
# Correlation Analysis: Feature Relationships
# Select key features for correlation heatmap
key_features = [
    'mfcc_0_mean', 'mfcc_1_mean', 'mfcc_2_mean',
    'spectral_centroid_mean', 'spectral_rolloff_mean',
    'f0_mean', 'voicing_ratio', 'energy_mean', 'zcr_mean'
]

correlation_matrix = df_features[key_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Audio Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Correlation heatmap saved")

## Section 5: Statistical Analysis of Feature Differences

In [ ]:
# Statistical tests: T-tests for genuine vs deepfake
from scipy.stats import ttest_ind

test_features = [
    'mfcc_0_mean', 'spectral_centroid_mean', 'f0_mean', 
    'voicing_ratio', 'energy_mean', 'zcr_mean'
]

results = []

print("Statistical Significance Tests (Independent T-tests)")
print("=" * 70)

for feature in test_features:
    genuine_vals = df_features[df_features['label'] == 'genuine'][feature]
    fake_vals = df_features[df_features['label'] == 'deepfake'][feature]
    
    t_stat, p_value = ttest_ind(genuine_vals, fake_vals)
    mean_diff = genuine_vals.mean() - fake_vals.mean()
    
    significant = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
    
    results.append({
        'Feature': feature,
        'Genuine_Mean': genuine_vals.mean(),
        'Deepfake_Mean': fake_vals.mean(),
        'Difference': mean_diff,
        'T-Statistic': t_stat,
        'P-Value': p_value,
        'Significant': significant
    })
    
    print(f"{feature:25s} | t={t_stat:7.2f} | p={p_value:.4f} {significant}")

df_stats = pd.DataFrame(results)
print("\n" + "=" * 70)
print(f"✓ Found {(df_stats['P-Value'] < 0.05).sum()} significantly different features")

## Section 6: Audio Augmentation Demonstration

In [ ]:
# Demonstration of augmentation techniques
sample_audio_path = str(audio_files[0])
y, sr = librosa.load(sample_audio_path, sr=SAMPLE_RATE)

# Apply augmentations
y_stretched = preprocessor.time_stretch(y, rate=1.1)
y_pitched = preprocessor.pitch_shift(y, steps=2)
y_noisy = preprocessor.add_noise(y, noise_factor=0.01)

# Visualize spectrograms
fig, axes = plt.subplots(4, 1, figsize=(14, 10))
fig.suptitle('Audio Augmentation Techniques - Spectrograms', fontsize=14, fontweight='bold')

audio_samples = [
    (y, 'Original'),
    (y_stretched, 'Time Stretched (1.1x)'),
    (y_pitched, 'Pitch Shifted (+2 steps)'),
    (y_noisy, 'Noise Added')
]

for idx, (audio, label) in enumerate(audio_samples):
    S = librosa.feature.melspectrogram(y=audio, sr=sr)
    S_db = librosa.power_to_db(S, ref=np.max)
    
    img = librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[idx])
    axes[idx].set_title(label, fontsize=12)
    fig.colorbar(img, ax=axes[idx], format='%+2.0f dB')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'audio_augmentation_spectrograms.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Audio augmentation spectrograms saved")

## Section 7: Generate Final Feature Matrix

In [ ]:
# Prepare feature matrix for ML training
# Separate features and labels
X = df_features.drop(['file', 'label', 'audio_path'], axis=1, errors='ignore')
y = df_features['label'].map({'genuine': 0, 'deepfake': 1}).values

print("Feature Matrix Summary")
print("=" * 50)
print(f"Shape: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"\nLabel Distribution:")
print(f"  Genuine (0): {(y == 0).sum()} samples")
print(f"  Deepfake (1): {(y == 1).sum()} samples")

# Check for missing values
missing_pct = (X.isnull().sum() / len(X) * 100)
if missing_pct.any():
    print(f"\nMissing values: {missing_pct[missing_pct > 0]}")
else:
    print("\nMissing values: None")

# Feature statistics
print(f"\nFeature Statistics:")
print(X.describe().round(3))

In [ ]:
# Save feature matrix for Phase 3 Step 3 (Baseline Models)
feature_output = OUTPUT_DIR / 'audio_features_matrix.csv'
df_export = X.copy()
df_export['label'] = y
df_export['label_name'] = df_features['label'].values
df_export['audio_file'] = df_features['file'].values

df_export.to_csv(feature_output, index=False)

print(f"✓ Feature matrix saved to: {feature_output}")
print(f"  Rows: {len(df_export)}")
print(f"  Columns: {len(df_export.columns)}")

# Also save the comprehensive statistics
stats_output = OUTPUT_DIR / 'statistical_tests_results.csv'
df_stats.to_csv(stats_output, index=False)
print(f"\n✓ Statistical test results saved to: {stats_output}")

## Section 8: Summary & Next Steps

In [ ]:
print("\n" + "="*70)
print("PHASE 3 STEP 2: AUDIO FEATURE ENGINEERING - COMPLETION SUMMARY")
print("="*70)

print("\n✓ DELIVERABLES COMPLETED:")
print("  1. Feature extraction pipeline (35+ audio features)")
print("     - MFCC coefficients (13 + deltas)")
print("     - Spectral features (centroid, rolloff, bandwidth)")
print("     - Prosodic features (F0, voicing ratio)")
print("     - Temporal features (duration, RMS, energy)")
print("  2. Preprocessing utilities (normalization, silence removal, augmentation)")
print("  3. EDA visualizations:")
print(f"     - MFCC distributions")
print(f"     - Spectral features boxplots")
print(f"     - Prosodic features (F0 + voicing)")
print(f"     - Feature correlation matrix")
print(f"     - Audio augmentation spectrograms")
print(f"  4. Feature matrix: {len(df_export)} samples × {len(df_export.columns)} attributes")
print(f"  5. Statistical analysis: {(df_stats['P-Value'] < 0.05).sum()} significant features")

print("\n💾 OUTPUT FILES SAVED:")
outputs = [
    'audio_features_matrix.csv',
    'statistical_tests_results.csv',
    'mfcc_distribution_comparison.png',
    'spectral_features_boxplot.png',
    'prosodic_features_distribution.png',
    'feature_correlation_matrix.png',
    'audio_augmentation_spectrograms.png'
]
for output in outputs:
    print(f"  ✓ {output}")

print("\n🎯 NEXT STEP: Phase 3 Step 3 - Baseline Model Training")
print("   Use audio_features_matrix.csv to train:")
print("   - Random Forest Classifier")
print("   - Logistic Regression")
print("   - SVM (RBF kernel)")
print("\n" + "="*70)